# Phase 3 — Step 1: build the subtype label table

**Scientific question (PHASE3_PLANNING §1).**
*Define the Phase 3 unit population, attach AIBS metamodel subtype labels with corrections, and publish the per-scan counts that gate every downstream LOSO decision.*

This notebook does **not** train any classifier. It produces:

1. The extended population table `data/processed/tables/units_v1_exc_best_subtype.parquet`, which is the Phase 1 canonical population (V1, excitatory, oracle-matched, best unit per neuron) with helper columns: `scan_id`, `subtype`, and binary indicators `is_L5_IT`, `is_L5_ET`, `is_L6_IT`, `is_L6_CT`.
2. Two count tables:
   - `phase3_subtype_counts_global.csv` — overall counts of the seven subtypes.
   - `phase3_subtype_counts_per_scan.csv` — per-scan crosstab, the input to LOSO validity.
3. `phase3_loso_valid_scans.json` — the *prespecified* lists of scans that are valid LOSO held-out folds for the L5 IT/ET and L6 IT/CT binaries, under the rule "minority class ≥ 5 cells AND both classes present in the held-out scan."

**Why a separate Step 1.** Per-scan counts are not just a sanity check — they decide which scans can serve as LOSO held-out folds. Locking the validity list now, before any classifier is fitted, removes the risk of post-hoc valid-scan selection in Steps 6 and 8.


## 1. Setup

In [1]:
from __future__ import annotations

import sys, json
from pathlib import Path

import numpy as np
import pandas as pd

# Repo root convention: notebooks/ is one level below repo root.
# Phase 3 notebooks live in notebooks/phase3/, so we go two levels up.
REPO_ROOT = Path('..').resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_RAW       = REPO_ROOT / 'data' / '1718' / 'raw'
DATA_TABLES    = REPO_ROOT / 'data' / 'processed' / 'tables'
DATA_TABLES.mkdir(parents=True, exist_ok=True)

UNITS_BEST_PATH        = DATA_TABLES / 'units_v1_exc_best.parquet'
AIBS_CT_PATH           = DATA_RAW / 'aibs_metamodel_celltypes_v661.csv'
AIBS_CT_CORR_PATH      = DATA_RAW / 'aibs_metamodel_celltypes_v661_corrections.csv'

UNITS_SUBTYPE_OUT      = DATA_TABLES / 'units_v1_exc_best_subtype.parquet'
COUNTS_GLOBAL_OUT      = DATA_TABLES / 'phase3_subtype_counts_global.csv'
COUNTS_PER_SCAN_OUT    = DATA_TABLES / 'phase3_subtype_counts_per_scan.csv'
LOSO_VALID_OUT         = DATA_TABLES / 'phase3_loso_valid_scans.json'

# The seven subtype labels in scope for Phase 3.
PHASE3_SUBTYPES = ['23P', '4P', '5P-IT', '5P-ET', '5P-NP', '6P-IT', '6P-CT']

# LOSO validity rule (PHASE3_PLANNING §2.5).
LOSO_MIN_MINORITY_CELLS = 5

print('repo root :', REPO_ROOT)
print('units in  :', UNITS_BEST_PATH)
print('aibs ct   :', AIBS_CT_PATH)
print('aibs corr :', AIBS_CT_CORR_PATH)


repo root : /Users/katiarusso/Documents/VS CODE/Classes/Neuroscience/Final Project
units in  : /Users/katiarusso/Documents/VS CODE/Classes/Neuroscience/Final Project/data/processed/tables/units_v1_exc_best.parquet
aibs ct   : /Users/katiarusso/Documents/VS CODE/Classes/Neuroscience/Final Project/data/1718/raw/aibs_metamodel_celltypes_v661.csv
aibs corr : /Users/katiarusso/Documents/VS CODE/Classes/Neuroscience/Final Project/data/1718/raw/aibs_metamodel_celltypes_v661_corrections.csv


## 2. Load the Phase 1 canonical population

In [2]:
units = pd.read_parquet(UNITS_BEST_PATH)
print('units_v1_exc_best.parquet shape:', units.shape)
print('columns:', list(units.columns))
print()
print('classification_system unique:', units['classification_system'].unique())
print('brain_area unique           :', units['brain_area'].unique())
print('tuning_type unique          :', units['tuning_type'].unique())
print()
print('cell_type value counts (raw, in cached file):')
print(units['cell_type'].value_counts(dropna=False).to_string())


units_v1_exc_best.parquet shape: (8910, 16)
columns: ['nucleus_id', 'pt_root_id', 'pt_position_x', 'pt_position_y', 'pt_position_z', 'classification_system', 'cell_type', 'brain_area', 'strategy_axon', 'strategy_dendrite', 'session', 'scan_idx', 'unit_id', 'cc_abs', 'tuning_type', 'layer']

classification_system unique: ['excitatory_neuron']
brain_area unique           : ['V1']
tuning_type unique          : ['matched']

cell_type value counts (raw, in cached file):
cell_type
23P      4213
4P       2845
5P-IT    1169
5P-ET     320
6P-IT     216
6P-CT     121
5P-NP      26


## 3. Verify cached `cell_type` matches corrections-applied AIBS metamodel

The cached `cell_type` column in `units_v1_exc_best.parquet` should already incorporate the corrections file. We re-derive the corrected metamodel here and confirm 100 % agreement; if it ever drifts in a future data refresh, this assertion is the early warning.


In [3]:
ct_base = pd.read_csv(AIBS_CT_PATH)[['pt_root_id', 'cell_type', 'classification_system']]
ct_corr = pd.read_csv(AIBS_CT_CORR_PATH)[['pt_root_id', 'cell_type', 'classification_system']]

# Dedup keeping last (latest) call per pt_root_id.
ct_base = ct_base.drop_duplicates('pt_root_id', keep='last').set_index('pt_root_id')
ct_corr = ct_corr.drop_duplicates('pt_root_id', keep='last').set_index('pt_root_id')

# Apply corrections: override base with corrections, then add corrections-only ids.
combined = ct_base.copy()
shared = ct_corr.index.intersection(combined.index)
combined.loc[shared, ['cell_type', 'classification_system']] = ct_corr.loc[shared].values
new_ids = ct_corr.index.difference(combined.index)
combined = pd.concat([combined, ct_corr.loc[new_ids]])

print('AIBS metamodel (corrections-applied) unique pt_root_id:', len(combined))

# Sanity check against the cached units file.
joined = units.merge(
    combined.reset_index().rename(columns={'cell_type': 'cell_type_metamodel'}),
    on='pt_root_id', how='left',
)
n_matched = joined['cell_type_metamodel'].notna().sum()
n_agree   = (joined['cell_type'] == joined['cell_type_metamodel']).sum()
print(f'units matched in metamodel : {n_matched} / {len(units)}')
print(f'cell_type agreement        : {n_agree} / {len(units)}')
assert n_matched == len(units), 'some units have no AIBS metamodel call'
assert n_agree   == len(units), 'cached cell_type disagrees with corrections-applied metamodel'
print('OK — cached cell_type already incorporates corrections.')


AIBS metamodel (corrections-applied) unique pt_root_id: 91489
units matched in metamodel : 8910 / 8910
cell_type agreement        : 8910 / 8910
OK — cached cell_type already incorporates corrections.


## 4. Build the extended Phase 3 unit table

In [4]:
# Verify cell_type values are exactly the seven Phase 3 subtypes.
unique_subtypes = set(units['cell_type'].unique())
unexpected = unique_subtypes - set(PHASE3_SUBTYPES)
missing    = set(PHASE3_SUBTYPES) - unique_subtypes
print('unexpected cell_type values:', unexpected)
print('missing cell_type values   :', missing)
assert not unexpected, 'unexpected cell_type values present'

# Build helper columns.
df = units.copy()
df['scan_id'] = df['session'].astype(int).astype(str) + '_' + df['scan_idx'].astype(int).astype(str)
df['subtype'] = df['cell_type']  # alias

df['is_L5_IT'] = (df['subtype'] == '5P-IT').astype('int8')
df['is_L5_ET'] = (df['subtype'] == '5P-ET').astype('int8')
df['is_L6_IT'] = (df['subtype'] == '6P-IT').astype('int8')
df['is_L6_CT'] = (df['subtype'] == '6P-CT').astype('int8')

print('extended table shape :', df.shape)
print('columns added        :', ['scan_id', 'subtype', 'is_L5_IT', 'is_L5_ET', 'is_L6_IT', 'is_L6_CT'])
print()
print('# unique scans       :', df['scan_id'].nunique())
print('# unique sessions    :', df['session'].nunique())


unexpected cell_type values: set()
missing cell_type values   : set()
extended table shape : (8910, 22)
columns added        : ['scan_id', 'subtype', 'is_L5_IT', 'is_L5_ET', 'is_L6_IT', 'is_L6_CT']

# unique scans       : 13
# unique sessions    : 6


## 5. Global subtype counts

In [5]:
global_counts = (
    df['subtype']
    .value_counts()
    .reindex(PHASE3_SUBTYPES)   # fixed display order
    .rename_axis('subtype')
    .to_frame('n')
)
global_counts['pct'] = (100.0 * global_counts['n'] / global_counts['n'].sum()).round(2)

# Layer cross-tab for context (depth-bin layer × metamodel subtype).
layer_x_subtype = pd.crosstab(df['layer'], df['subtype']).reindex(columns=PHASE3_SUBTYPES, fill_value=0)

print('=== global subtype counts (Phase 3 population, n =', len(df), ') ===')
print(global_counts.to_string())
print()
print('=== depth-bin layer × subtype (sanity) ===')
print(layer_x_subtype.to_string())


=== global subtype counts (Phase 3 population, n = 8910 ) ===
            n    pct
subtype             
23P      4213  47.28
4P       2845  31.93
5P-IT    1169  13.12
5P-ET     320   3.59
5P-NP      26   0.29
6P-IT     216   2.42
6P-CT     121   1.36

=== depth-bin layer × subtype (sanity) ===
subtype   23P    4P  5P-IT  5P-ET  5P-NP  6P-IT  6P-CT
layer                                                 
L1         15     0      0      0      0      0      0
L2/3     4160    87      0      0      0      0      0
L4         38  2600     28      4      0      0      0
L5          0   158   1117    289     18     20     13
L6          0     0     24     27      8    196    108


## 6. Per-scan crosstab

In [6]:
per_scan = pd.crosstab(df['scan_id'], df['subtype']).reindex(columns=PHASE3_SUBTYPES, fill_value=0)
per_scan['total'] = per_scan.sum(axis=1)
per_scan = per_scan.sort_values('total', ascending=False)
print('=== per-scan subtype counts ===')
print(per_scan.to_string())


=== per-scan subtype counts ===
subtype  23P   4P  5P-IT  5P-ET  5P-NP  6P-IT  6P-CT  total
scan_id                                                    
6_4      473  175    206      5      2     41     14    916
6_7      455  169    183     33      2     22     12    876
9_3        2  815      0      1      0      0      0    818
8_5      400  268     53     56      3      0      0    780
6_2      282  292    150      9      5     27      9    774
4_7      377  240     39     66      5      0      1    728
5_7      185  284    134     16      1     57     24    701
6_6      401   87    158     16      1     23     11    697
5_6      151  249    146     44      0     46     50    686
9_4      680    0      0      0      0      0      0    680
7_3      203  190     92     64      3      0      0    552
9_6      428    0      0      0      0      0      0    428
7_5      176   76      8     10      4      0      0    274


## 7. LOSO validity — prespecified

A held-out scan is *valid* for a binary task if both classes are present **and** the minority class has at least `LOSO_MIN_MINORITY_CELLS` cells in that scan. We commit this rule **before** any classifier is fitted (PHASE3_PLANNING §2.5).


In [7]:
def loso_validity(per_scan_table: pd.DataFrame, class_a: str, class_b: str,
                  min_minority: int = LOSO_MIN_MINORITY_CELLS) -> pd.DataFrame:
    # For each scan, decide whether it is a valid LOSO held-out fold
    # for the binary class_a vs class_b. Returns the per-scan table
    # with both_present, minority_n, valid columns added.
    sub = per_scan_table[[class_a, class_b]].copy()
    sub.columns = [class_a, class_b]
    sub['both_present'] = (sub[class_a] > 0) & (sub[class_b] > 0)
    sub['minority_n']   = sub[[class_a, class_b]].min(axis=1)
    sub['valid']        = sub['both_present'] & (sub['minority_n'] >= min_minority)
    return sub


val_l5_it_et = loso_validity(per_scan, '5P-IT', '5P-ET')
val_l6_it_ct = loso_validity(per_scan, '6P-IT', '6P-CT')

print('=== L5 IT vs ET — per-scan validity ===')
print(val_l5_it_et.to_string())
print()
print('=== L6 IT vs CT — per-scan validity ===')
print(val_l6_it_ct.to_string())
print()
print('valid scans for L5 IT/ET:', val_l5_it_et.index[val_l5_it_et['valid']].tolist())
print('valid scans for L6 IT/CT:', val_l6_it_ct.index[val_l6_it_ct['valid']].tolist())


=== L5 IT vs ET — per-scan validity ===
         5P-IT  5P-ET  both_present  minority_n  valid
scan_id                                               
6_4        206      5          True           5   True
6_7        183     33          True          33   True
9_3          0      1         False           0  False
8_5         53     56          True          53   True
6_2        150      9          True           9   True
4_7         39     66          True          39   True
5_7        134     16          True          16   True
6_6        158     16          True          16   True
5_6        146     44          True          44   True
9_4          0      0         False           0  False
7_3         92     64          True          64   True
9_6          0      0         False           0  False
7_5          8     10          True           8   True

=== L6 IT vs CT — per-scan validity ===
         6P-IT  6P-CT  both_present  minority_n  valid
scan_id                                

## 8. Save outputs

In [8]:
# 8.1 Extended unit table
df.to_parquet(UNITS_SUBTYPE_OUT, index=False)
print(f'wrote {UNITS_SUBTYPE_OUT}  ({UNITS_SUBTYPE_OUT.stat().st_size / 1024:.1f} KB)')

# 8.2 Global counts CSV
global_counts.to_csv(COUNTS_GLOBAL_OUT)
print(f'wrote {COUNTS_GLOBAL_OUT}')

# 8.3 Per-scan counts CSV
per_scan.to_csv(COUNTS_PER_SCAN_OUT)
print(f'wrote {COUNTS_PER_SCAN_OUT}')

# 8.4 LOSO validity JSON (locked, prespecified)
loso_payload = {
    'rule': {
        'min_minority_cells': LOSO_MIN_MINORITY_CELLS,
        'both_classes_required': True,
        'frozen_at_step': 'phase3_step1',
    },
    'L5_IT_vs_ET': {
        'class_a': '5P-IT',
        'class_b': '5P-ET',
        'valid_scans':   val_l5_it_et.index[val_l5_it_et['valid']].tolist(),
        'invalid_scans': val_l5_it_et.index[~val_l5_it_et['valid']].tolist(),
        'per_scan': val_l5_it_et.reset_index().to_dict(orient='records'),
    },
    'L6_IT_vs_CT': {
        'class_a': '6P-IT',
        'class_b': '6P-CT',
        'valid_scans':   val_l6_it_ct.index[val_l6_it_ct['valid']].tolist(),
        'invalid_scans': val_l6_it_ct.index[~val_l6_it_ct['valid']].tolist(),
        'per_scan': val_l6_it_ct.reset_index().to_dict(orient='records'),
    },
    'global_counts': global_counts['n'].to_dict(),
}

with open(LOSO_VALID_OUT, 'w') as f:
    json.dump(loso_payload, f, indent=2, default=int)
print(f'wrote {LOSO_VALID_OUT}')


wrote /Users/katiarusso/Documents/VS CODE/Classes/Neuroscience/Final Project/data/processed/tables/units_v1_exc_best_subtype.parquet  (493.6 KB)
wrote /Users/katiarusso/Documents/VS CODE/Classes/Neuroscience/Final Project/data/processed/tables/phase3_subtype_counts_global.csv
wrote /Users/katiarusso/Documents/VS CODE/Classes/Neuroscience/Final Project/data/processed/tables/phase3_subtype_counts_per_scan.csv
wrote /Users/katiarusso/Documents/VS CODE/Classes/Neuroscience/Final Project/data/processed/tables/phase3_loso_valid_scans.json


## 9. Summary — what Steps 2–9 inherit

In [9]:
print('=== PHASE 3 STEP 1 SUMMARY ===')
print(f'population        : V1 / excitatory / oracle-matched / best (n = {len(df)})')
print(f'unique scans      : {df["scan_id"].nunique()}')
print()
print('global subtype counts:')
for s in PHASE3_SUBTYPES:
    print(f'  {s:<6s} {int(global_counts.loc[s, "n"]):>5d}')
print()
n_l5_pair = int((df["subtype"].isin(["5P-IT", "5P-ET"])).sum())
n_l6_pair = int((df["subtype"].isin(["6P-IT", "6P-CT"])).sum())
print(f'L5 IT/ET binary pop  : {n_l5_pair} (5P-IT={int(global_counts.loc["5P-IT", "n"])}, 5P-ET={int(global_counts.loc["5P-ET", "n"])})')
print(f'L6 IT/CT binary pop  : {n_l6_pair} (6P-IT={int(global_counts.loc["6P-IT", "n"])}, 6P-CT={int(global_counts.loc["6P-CT", "n"])})')
print()
print(f'LOSO valid scans, L5 IT/ET ({len(loso_payload["L5_IT_vs_ET"]["valid_scans"])}):',
      loso_payload['L5_IT_vs_ET']['valid_scans'])
print(f'LOSO valid scans, L6 IT/CT ({len(loso_payload["L6_IT_vs_CT"]["valid_scans"])}):',
      loso_payload['L6_IT_vs_CT']['valid_scans'])
print()
print('Outputs ready for Step 2 (L5 IT vs ET main experiment).')


=== PHASE 3 STEP 1 SUMMARY ===
population        : V1 / excitatory / oracle-matched / best (n = 8910)
unique scans      : 13

global subtype counts:
  23P     4213
  4P      2845
  5P-IT   1169
  5P-ET    320
  5P-NP     26
  6P-IT    216
  6P-CT    121

L5 IT/ET binary pop  : 1489 (5P-IT=1169, 5P-ET=320)
L6 IT/CT binary pop  : 337 (6P-IT=216, 6P-CT=121)

LOSO valid scans, L5 IT/ET (10): ['6_4', '6_7', '8_5', '6_2', '4_7', '5_7', '6_6', '5_6', '7_3', '7_5']
LOSO valid scans, L6 IT/CT (6): ['6_4', '6_7', '6_2', '5_7', '6_6', '5_6']

Outputs ready for Step 2 (L5 IT vs ET main experiment).
